In [7]:
from transformers import AutoTokenizer
import sentencepiece as spm
from datasets import load_dataset

Pretrained GPT2 BPE tokenizer

In [8]:
gpt_bpe_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [9]:
text = "Byte Pair Encoding is used in many language models."

# Text -> list of token strings
tokens = gpt_bpe_tokenizer.tokenize(text)
print(tokens)

# Text -> list of token IDs
ids = gpt_bpe_tokenizer.encode(text)
print(ids)

# Token IDs -> text
print(gpt_bpe_tokenizer.decode(ids))

# Size of the vocabulary
print(gpt_bpe_tokenizer.vocab_size)

['Byte', 'ĠPair', 'ĠEnc', 'oding', 'Ġis', 'Ġused', 'Ġin', 'Ġmany', 'Ġlanguage', 'Ġmodels', '.']
[40778, 39645, 14711, 7656, 318, 973, 287, 867, 3303, 4981, 13]
Byte Pair Encoding is used in many language models.
50257


In [10]:
# Load WikiText-103 (raw version, no loading script needed)
ds = load_dataset("iohadrubin/wikitext-103-raw-v1")

# Drop empty lines; each non-empty line is a paragraph or a heading
# train_texts = [t for t in ds["train"]["text"] if t.strip()]
# test_texts = [t for t in ds["test"]["text"] if t.strip()]

# print(len(train_texts), len(test_texts))

Whitespace-based tokenizer (custom implementation)

In [11]:
import os

from custom_tokenizer import WhitespaceTokenizer

WS_PATH = "ws_tokenizer.json"

if os.path.exists(WS_PATH):
    # Reuse the vocabulary saved by a previous run
    ws_tokenizer = WhitespaceTokenizer.load(WS_PATH)
else:
    # Same N as the GPT-2 BPE tokenizer (includes the <UNK> token)
    ws_tokenizer = WhitespaceTokenizer(vocab_size=gpt_bpe_tokenizer.vocab_size)
    ws_tokenizer.train(t for t in ds["train"]["text"] if t.strip())
    ws_tokenizer.save(WS_PATH)

text = "Byte Pair Encoding is used in many language models."

# Text -> list of token strings (punctuation is a separate token)
print(ws_tokenizer.tokenize(text))

# Text -> list of token IDs (out-of-vocabulary tokens map to <UNK>, id 0)
ids = ws_tokenizer.encode(text)
print(ids)

# Token IDs -> text
print(ws_tokenizer.decode(ids))

# Size of the vocabulary
print(len(ws_tokenizer.token_to_id))

['byte', 'pair', 'encoding', 'is', 'used', 'in', 'many', 'language', 'models', '.']
[42061, 1446, 34148, 23, 102, 6, 112, 936, 2260, 3]
byte pair encoding is used in many language models.
50257


SentencePiece tokenizer

In [12]:
# Train on the same data as the other tokenizers (WikiText-103 train), streamed straight
# from the dataset so no corpus file is needed. Rows are whole articles, so split them
# into lines: SentencePiece skips sentences longer than max_sentence_length (4192 bytes)
def train_lines():
    for article in ds["train"]["text"]:
        for line in article.split("\n"):
            if line.strip():
                yield line.strip()


# Training writes sp.model and sp.vocab, so skip it when they already exist
if not os.path.exists("sp.model"):
    spm.SentencePieceTrainer.train(
        sentence_iterator=train_lines(),
        model_prefix="sp",  # creates sp.model and sp.vocab
        vocab_size=gpt_bpe_tokenizer.vocab_size,  # same N as the other tokenizers
        model_type="unigram",  # or "bpe"
        input_sentence_size=300_000,  # sample of lines, keeps memory and time reasonable
        shuffle_input_sentence=True,
    )

# Load the trained model
sp = spm.SentencePieceProcessor(model_file="sp.model")

text = "Byte Pair Encoding is used in many language models."
print(sp.encode(text, out_type=str))  # pieces, e.g. ['▁Byte', '▁Pair', ...]
ids = sp.encode(text, out_type=int)  # token ids
print(ids)
print(sp.decode(ids))  # lossless round trip
print(sp.get_piece_size())  # vocabulary size, should be 50257

['▁By', 'te', '▁P', 'air', '▁En', 'coding', '▁is', '▁used', '▁in', '▁many', '▁language', '▁models', '.']
[276, 869, 446, 3556, 4852, 26714, 24, 110, 10, 143, 1249, 2796, 5]
Byte Pair Encoding is used in many language models.
50257


Language model training (same GPT-2 architecture, only the tokenizer differs)

In [ ]:
from train_lm import TrainConfig, encode_corpus, train

config = TrainConfig(vocab_size=gpt_bpe_tokenizer.vocab_size)


def lines(split):
    return [line for article in ds[split]["text"] for line in article.split("\n") if line.strip()]


# Every tokenizer encodes exactly the same text
train_texts = lines("train")[:200_000]
val_texts = lines("validation")
val_chars = sum(len(t) for t in val_texts)

tokenizers = {
    "bpe": lambda t: gpt_bpe_tokenizer.encode(t),
    "whitespace": ws_tokenizer.encode,
    "sentencepiece": lambda t: sp.encode(t),
}

# Encode once and cache the ids on disk
encoded = {}
for name, encode in tokenizers.items():
    tr = encode_corpus(encode, train_texts, f"ids_{name}_train.npy")
    va = encode_corpus(encode, val_texts, f"ids_{name}_val.npy")
    encoded[name] = (tr, va)
    print(f"{name}: {len(tr):,} train tokens, {len(va):,} validation tokens")

bpe: 19,131,334 train tokens, 235,505 validation tokens
whitespace: 17,494,686 train tokens, 216,580 validation tokens
sentencepiece: 18,345,745 train tokens, 226,497 validation tokens


: 

In [ ]:
# Run a short test first (e.g. config.max_steps = 20) to measure the time per step
for name, (tr, va) in encoded.items():
    train(name, tr, va, val_chars, config)

bpe: 30.0M parameters on cpu


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


step 0: val loss 10.897 nats/token, 3.368 bits/char, 127s
